In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("recipes_data.csv")
df.head()

,title,ingredients,directions,link,source,NER,site
0,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...",www.cookbooks.com/Recipe-Details.aspx?id=44874,Gathered,"[""bite size shredded rice biscuits"", ""vanilla""...",www.cookbooks.com
1,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....",www.cookbooks.com/Recipe-Details.aspx?id=699419,Gathered,"[""cream of mushroom soup"", ""beef"", ""sour cream...",www.cookbooks.com
2,Creamy Corn,"[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg...","[""In a slow cooker, combine all ingredients. C...",www.cookbooks.com/Recipe-Details.aspx?id=10570,Gathered,"[""frozen corn"", ""pepper"", ""cream cheese"", ""gar...",www.cookbooks.com
3,Chicken Funny,"[""1 large whole chicken"", ""2 (10 1/2 oz.) cans...","[""Boil and debone chicken."", ""Put bite size pi...",www.cookbooks.com/Recipe-Details.aspx?id=897570,Gathered,"[""chicken gravy"", ""cream of mushroom soup"", ""c...",www.cookbooks.com
4,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker ...","[""Combine first four ingredients and press in ...",www.cookbooks.com/Recipe-Details.aspx?id=659239,Gathered,"[""graham cracker crumbs"", ""powdered sugar"", ""p...",www.cookbooks.com


In [11]:
df.shape

(2231142, 7)

In [15]:
import ast

# Parse the string lists into actual lists, then explode and get unique values
all_ingredients = df['NER'].dropna().apply(ast.literal_eval).explode().unique()

print(f"Total unique ingredients: {len(all_ingredients)}")
print(all_ingredients)

Total unique ingredients: 198900
['bite size shredded rice biscuits' 'vanilla' 'brown sugar' ...
 'white string cheese' 'stone-grnd' 'five-cheese']


In [ ]:
ingredients = pd.DataFrame(all_ingredients)
data = ingredients.to_csv("ingredients.csv", index = False)



In [ ]:
sample = df['NER'].sample(2000000)

all_ingredients_sample = sample.dropna().apply(ast.literal_eval).explode().unique()


array(['blackberry', 'blueberries', 'pineapple', ...,
       'freshly chopped onions', 'watermelon fruit',
       'commercial biscotti'], dtype=object)

In [7]:
df_canonical_ingredients = pd.read_csv("canonical_ingredients.csv")
df_canonical_ingredients.head()

,id,canonical_ingredient,variant_count
0,1,abalone,13
1,2,adzuki beans,23
2,3,agave,16
3,4,aioli,45
4,5,alfredo sauce,73


In [12]:
df_canonical_ingredients.shape

(597, 3)

In [8]:
df_recipe_filtered = pd.read_csv("recipes_data_filtered.csv")
df_recipe_filtered.head() 

,title,ingredients,directions,link,site,coverage,NER_canonical
0,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...",www.cookbooks.com/Recipe-Details.aspx?id=44874,www.cookbooks.com,0.833333,"['rice', 'vanilla', 'brown sugar', '', 'milk',..."
1,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....",www.cookbooks.com/Recipe-Details.aspx?id=699419,www.cookbooks.com,0.750000,"['cream of mushroom soup', '', 'sour cream', '..."
2,Creamy Corn,"[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg...","[""In a slow cooker, combine all ingredients. C...",www.cookbooks.com/Recipe-Details.aspx?id=10570,www.cookbooks.com,0.666667,"['', '', 'cream cheese', 'garlic powder', 'but..."
3,Chicken Funny,"[""1 large whole chicken"", ""2 (10 1/2 oz.) cans...","[""Boil and debone chicken."", ""Put bite size pi...",www.cookbooks.com/Recipe-Details.aspx?id=897570,www.cookbooks.com,0.250000,"['', 'cream of mushroom soup', '', '']"
4,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker ...","[""Combine first four ingredients and press in ...",www.cookbooks.com/Recipe-Details.aspx?id=659239,www.cookbooks.com,1.000000,"['graham cracker crumbs', 'powdered sugar', 'p..."


In [13]:
df_recipe_filtered.shape

(2231142, 7)

In [17]:
canonical = pd.read_csv("canonical_ingredients.csv")
canonical = canonical.rename(columns={"id": "ingredientId", "canonical_ingredient": "name"})
canonical["restrictions"] = None

canonical[["ingredientId", "name", "restrictions"]].to_csv("ingredients_export.csv", index=False)
print(f"Exported {len(canonical)} ingredients")

Exported 597 ingredients


In [18]:
import ast

def safe_parse(val):
    try:
        return ast.literal_eval(val) if isinstance(val, str) else []
    except:
        return []

name_to_id = dict(zip(canonical["name"], canonical["ingredientId"]))

recipes = pd.read_csv("recipes_data_filtered.csv")
recipes = recipes[recipes["coverage"] >= 0.50].reset_index(drop=True)

recipes["NER_canonical"] = recipes["NER_canonical"].apply(safe_parse)
recipes["ingredients"]   = recipes["ingredients"].apply(safe_parse)
recipes["directions"]    = recipes["directions"].apply(safe_parse)

recipes["ingredient_ids"] = recipes["NER_canonical"].apply(
    lambda lst: [name_to_id[n.strip().lower()] for n in lst if n.strip().lower() in name_to_id]
)

recipes[["title", "ingredient_ids", "ingredients", "directions", "link"]].rename(columns={
    "title":      "name",
    "ingredients": "unfiltered_ingredients",
    "directions":  "instructions",
    "link":        "website"
}).to_csv("recipes_export.csv", index=False)

print(f"Exported {len(recipes)} recipes")

Exported 2195357 recipes


In [20]:
recipe_export = pd.read_csv("recipes_export.csv")
recipe_export.head

<bound method NDFrame.head of                                                   name  \
0                                  No-Bake Nut Cookies   
1                                Jewell Ball'S Chicken   
2                                          Creamy Corn   
3                                 Reeses Cups(Candy)     
4                             Cheeseburger Potato Soup   
...                                                ...   
2195352                            Sunny's Fake Crepes   
2195353                                     Devil Eggs   
2195354  Extremely Easy and Quick - Namul Daikon Salad   
2195355     Pan-Roasted Pork Chops With Apple Fritters   
2195356                 Polpette in Spicy Tomato Sauce   

                                            ingredient_ids  \
0                                  [448, 553, 76, 331, 81]   
1                                          [163, 497, 115]   
2                                      [160, 223, 81, 470]   
3                        